# Agent中使用短期记忆(Memory)

## 传统的方式

以React方式为例

In [1]:
import dotenv
from langchain.chains.summarize.refine_prompts import prompt_template
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
import os

from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, create_tool_calling_agent, create_react_agent, AgentExecutor
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.memory import ConversationBufferMemory

#1、提供搜索的工具

dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")  # 需要替换为你的 Tavily API 密钥


search = TavilySearchResults(max_results=2)

search_tool = Tool(
    name="search_tool",
    func=search.run,
    description="用于互联网新闻的检索"
)

#2、提供工具集
tools = [search_tool]

#3、提供大模型
dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#4、提供记忆模块：ConversationBufferMemory，作用是存储对话历史记录
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
)

#6、提供AgentExecutor
agent_executor = initialize_agent(
    #5、提供Agent
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    llm=llm,
    tools=tools,
    verbose=True,
    memory=memory,  #添加memory
)

#7、调用
# 因为我们使用的是Agent的提示词模板是默认的。所以我们需要知道默认的提示词模板中的一些变量。
agent_executor.invoke({"input": "明天北京的天气怎么样?"})


C:\Users\41507\AppData\Local\Temp\ipykernel_24220\2486841372.py:39: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(
C:\Users\41507\AppData\Local\Temp\ipykernel_24220\2486841372.py:45: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use :meth:`~Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc.` instead.
  agent_executor = initialize_agent(




> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: search_tool
Action Input: "北京 明天的天气预报"
```
Observation: [{'url': 'https://www.bjfsh.gov.cn/bsfw2/tdrqfw/cjr/qxyztq/202311/t20231103_40068148.shtml', 'content': '* 政务服务 首页\xa0>\xa0政务服务\xa0>\xa0特色服务\xa0>\xa0气象服务\xa0>\xa0一周天气 日期:2023-11-03 16:47\xa0\xa0\xa0\xa0 来源:区气象局 **(****11****月****4****日～****11****月****10****日)** | **日期** | **天气** | **气温(℃)** | **风向** | **风力** | | 夜间 | 多云转阴，有零星小雨 | 8 | 南转北风 | 1、2级 | | 夜间 | 阴转多云 | 3 | 偏北风 | 4级左右 | | 6日 | 白天 | 多云转晴 | 11 | 偏北风 | 4级左右转2级 | | 7日 | 白天 | 晴转多云 | 13 | 北转南风 | 2、3级 | | 8日 | 白天 | 晴 | 12 | 偏北风 | 3、4级 | | 夜间 | 晴 | -2 | 偏北风 | 1、2级 | | 夜间 | 晴间多云 | 1 | 南转北风 | 1、2级 | | 10日 | 白天 | 晴转多云 | 11 | 北转南风 | 2、3级 |'}, {'url': 'https://www.weather.com.cn/weather/101010100.shtml', 'content': '15/*4℃* 19/*3℃* 14/*1℃* 13/*4℃* 15/*6℃* 14/*7℃* 15/*6℃* * 适宜 *洗车指数* * 强 *紫外线指数* 涂擦SPF大于15、PA+防晒护肤品。 * 适宜 *运动指数* * 适宜 *洗车指数* * 强 *紫外线指数* 涂擦SPF大于15、PA+防晒护肤品。 * 适宜 *洗车指数* * 强 *紫外线指数* 涂擦SP

{'input': '明天北京的天气怎么样?',
 'chat_history': [HumanMessage(content='明天北京的天气怎么样?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='明天（11月4日）北京的天气预报是多云转阴，有零星小雨，夜间气温在8℃左右，风向为南转北风，风力为1到2级。白天气温预计在11℃左右，风力会有所减弱。请注意适时增添衣物以应对降温和降雨。\n```', additional_kwargs={}, response_metadata={})],
 'output': '明天（11月4日）北京的天气预报是多云转阴，有零星小雨，夜间气温在8℃左右，风向为南转北风，风力为1到2级。白天气温预计在11℃左右，风力会有所减弱。请注意适时增添衣物以应对降温和降雨。\n```'}

In [2]:
agent_executor.invoke({"input": "上海呢？"})



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: search_tool
Action Input: 上海 11月4日 天气预报
```
Observation: [{'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/november-weather/106577', 'content': '返回  # 上海, 上海市 63°F    63° 上海, 上海市 天氣 今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動 ### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊 今天   每小時   每天   雷達   MinuteCast®  ## 每月  空氣品質   健康與活動 ## 十一月 一月   二月   三月   四月   五月   六月   七月   八月   九月   十月   十一月   十二月 ## 每天 65° 58° 27 67° 55° 28 67° 53° 29 70° 53° 30 71° 60° 31 73° 60° 1 70° 57° 2 無 3 65° 51° 4 67° 55° 5 73° 62° 6 74° 63° 7 74° 64° 8 76° 62° 9 72° 60° 10 62° 52° 11 61° ### 十二月 2025 ### 一月 2026 ### 二月 2026 世界 亞洲 中國 上海市 上海 * 嘉興, 浙江省 * 寧波, 浙江省'}, {'url': 'http://sh.cma.gov.cn/sh/tqyb/qxsk/', 'content': '网站地图 \xa0|\xa0 联系我们\xa0|\xa0 加入收藏) 当前位置： 首页>\xa0上海市气象局>\xa0气象服务>\xa0天气实况 灾害天气预警 * 上海今日天气 * 上海中心气象台 >>返回 栏目导航 气象报告 今日天气 灾害天气预警 空气质量预报 五日天气预报 天气实况 卫星云图 雷达回波图 生活气象指数 高温情况 天气实况 卫星云图 雷达回波图 上海市生活气象指

{'input': '上海呢？',
 'chat_history': [HumanMessage(content='明天北京的天气怎么样?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='明天（11月4日）北京的天气预报是多云转阴，有零星小雨，夜间气温在8℃左右，风向为南转北风，风力为1到2级。白天气温预计在11℃左右，风力会有所减弱。请注意适时增添衣物以应对降温和降雨。\n```', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='上海呢？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='明天（11月4日）上海的天气预报是多云，白天气温预计在19℃左右，夜间气温降至12℃。风速较轻，适合外出活动。请根据天气情况适时增添衣物。', additional_kwargs={}, response_metadata={})],
 'output': '明天（11月4日）上海的天气预报是多云，白天气温预计在19℃左右，夜间气温降至12℃。风速较轻，适合外出活动。请根据天气情况适时增添衣物。'}

## 通用的方式

以FUNCTION_CALL方式为例

补充：通用方式，相较于传统方式，可以提供自定义的提示词模板

说明：
如果使用的是FUNCTION_CALL方式，则创建Agent时，只能使用ChatPromptTemplate

如果使用的是React方式，则创建Agent时，可以使用ChatPromptTemplate、PromptTemplate


In [3]:
import dotenv
from langchain.chains.summarize.refine_prompts import prompt_template
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
import os

from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, create_tool_calling_agent, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.memory import ConversationBufferMemory
from langchain.agents.agent import AgentExecutor

#1、提供搜索的工具
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")  # 需要替换为你的 Tavily API 密钥

search = TavilySearchResults(max_results=2)

search_tool = Tool(
    name="search_tool",
    func=search.run,
    description="用于互联网新闻的检索"
)

#2、提供工具集
tools = [search_tool]

#3、提供大模型
dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#4、提供记忆模块：ConversationBufferMemory
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
)
#5、提供提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "我是一个AI助手，可以调用工具，即时解决用户的问题"),
    ("human", "{question}"),
    ("system", "{agent_scratchpad}"), #agent_scratchpad作用是存储工具调用结果，scratchpad英文翻译是草稿纸
    ("system","{chat_history}") #与ConversationBufferMemory中的memory_key相同
])
#6、提供Agent
agent = create_tool_calling_agent(
    prompt=prompt_template,
    llm=llm,
    tools=tools,
)


#7、提供AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
)

#8、执行
agent_executor.invoke({"question": "北京明天的天气怎么样？"})




> Entering new AgentExecutor chain...

Invoking: `search_tool` with `北京 明天的天气`


[{'url': 'https://www.accuweather.com/en/cn/beijing/101924/november-weather/101924', 'content': '### Hurricane Tracker ### Severe Weather ### Radar & Maps ### Warnings ### Data Suite ### Forensics ### Advertising ### Superior Accuracy™ ### Hurricane Tracker ### Severe Weather ### Radar & Maps ### News ### Video Weather Forecasts 22 hours ago Weather News 2 hours ago Weather Forecasts Weather News 1 day ago Weather News For Business   For Partners   For Advertising   AccuWeather APIs   AccuWeather Connect  RealFeel® and RealFeel Shade™  Personal Weather Stations AccuWeather Ready   Business   Health   Hurricane   Leisure and Recreation   Severe Weather   Space and Astronomy   Sports   Travel   Weather News For Business   For Partners   For Advertising   AccuWeather APIs   AccuWeather Connect  RealFeel® and RealFeel Shade™  Personal Weather Stations AccuWeather Ready   Business   Health   Hurricane   Leisu

{'question': '北京明天的天气怎么样？',
 'chat_history': [HumanMessage(content='北京明天的天气怎么样？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='我无法直接获取明天北京的天气信息。你可以查看以下链接获取最新的天气预报：\n\n- [AccuWeather 北京天气](https://www.accuweather.com/en/cn/beijing/101924/november-weather/101924)\n- [国家气候中心](https://www.ncc-cma.net/climate/pred)\n\n请访问这些网站以获取详细的天气信息。', additional_kwargs={}, response_metadata={})],
 'output': '我无法直接获取明天北京的天气信息。你可以查看以下链接获取最新的天气预报：\n\n- [AccuWeather 北京天气](https://www.accuweather.com/en/cn/beijing/101924/november-weather/101924)\n- [国家气候中心](https://www.ncc-cma.net/climate/pred)\n\n请访问这些网站以获取详细的天气信息。'}

In [4]:
agent_executor.invoke({"question": "上海呢？"})



> Entering new AgentExecutor chain...

Invoking: `search_tool` with `上海天气预报`


[{'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/november-weather/106577', 'content': '返回  # 上海, 上海市 63°F    63° 上海, 上海市 天氣 今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動 ### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊 今天   每小時   每天   雷達   MinuteCast®  ## 每月  空氣品質   健康與活動 ## 十一月 一月   二月   三月   四月   五月   六月   七月   八月   九月   十月   十一月   十二月 ## 每天 65° 58° 27 67° 55° 28 67° 53° 29 70° 53° 30 71° 60° 31 73° 60° 1 70° 57° 2 無 3 65° 51° 4 67° 55° 5 73° 62° 6 74° 63° 7 74° 64° 8 76° 62° 9 72° 60° 10 62° 52° 11 61° ### 十二月 2025 ### 一月 2026 ### 二月 2026 世界 亞洲 中國 上海市 上海 * 嘉興, 浙江省 * 寧波, 浙江省'}, {'url': 'https://www.weather.com.cn/weather/101020100.shtml', 'content': '更多 :   北京 上海 成都 杭州 南京 天津 深圳 重庆 西安 广州 青岛 武汉 :   故宫 阳朔漓江 龙门石窟 野三坡 颐和园 九寨沟 东方明珠 凤凰古城 秦始皇陵 桃花源 :   佘山 春城湖畔 华彬庄园 观澜湖 依必朗 旭宝 博鳌 玉龙雪山 番禺南沙 东方明珠 <<返回 全国 全国> 上海 > 城区 * 较易发 *感冒指数* * 较易发 *过敏指数* * 较冷 *穿衣指数* * 少发 *感冒指数* * 较易发 *过敏指数* * 较舒适 *穿衣指数* * 少发 *感冒指

{'question': '上海呢？',
 'chat_history': [HumanMessage(content='北京明天的天气怎么样？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='我无法直接获取明天北京的天气信息。你可以查看以下链接获取最新的天气预报：\n\n- [AccuWeather 北京天气](https://www.accuweather.com/en/cn/beijing/101924/november-weather/101924)\n- [国家气候中心](https://www.ncc-cma.net/climate/pred)\n\n请访问这些网站以获取详细的天气信息。', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='上海呢？', additional_kwargs={}, response_metadata={}),
  AIMessage(content='关于上海的天气预报，你可以查看以下链接获取最新信息：\n\n- [AccuWeather 上海天气](https://www.accuweather.com/zh/cn/shanghai/106577/november-weather/106577)\n- [中国天气网 上海天气](https://www.weather.com.cn/weather/101020100.shtml)\n\n请访问这些网站以获取详细的天气信息。', additional_kwargs={}, response_metadata={})],
 'output': '关于上海的天气预报，你可以查看以下链接获取最新信息：\n\n- [AccuWeather 上海天气](https://www.accuweather.com/zh/cn/shanghai/106577/november-weather/106577)\n- [中国天气网 上海天气](https://www.weather.com.cn/weather/101020100.shtml)\n\n请访问这些网站以获取详细的天气信息。'}